In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import QuantileTransformer

import lightgbm as lgb
from xgboost import XGBClassifier


In [3]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [4]:
from catboost import CatBoostClassifier

In [12]:
train = pd.read_csv("/content/drive/MyDrive/ML Challenge Dataset/TRAIN.csv")
test = pd.read_csv("/content/drive/MyDrive/ML Challenge Dataset/TEST.csv")



In [13]:
train = train.drop_duplicates().reset_index(drop=True)

In [14]:
train["is_train"] = 1
test["is_train"] = 0
test["Class"] = -1

df = pd.concat([train,test],axis=0).reset_index(drop=True)

In [15]:
pairs = [
("F01","F29"),
("F02","F22"),
("F03","F23"),
("F04","F24"),
("F05","F25"),
("F06","F26"),
("F07","F27"),
("F08","F28")
]

for a,b in pairs:

    if a in df.columns and b in df.columns:

        df[f"{a}_{b}_diff"] = df[a] - df[b]
        df[f"{a}_{b}_ratio"] = df[a] / (df[b] + 1e-6)
        df[f"{a}_{b}_sum"] = df[a] + df[b]

In [16]:
feature_cols = [c for c in df.columns if c not in ["Class","is_train"]]

df["row_mean"] = df[feature_cols].mean(axis=1)
df["row_std"] = df[feature_cols].std(axis=1)
df["row_max"] = df[feature_cols].max(axis=1)
df["row_min"] = df[feature_cols].min(axis=1)
df["row_range"] = df["row_max"] - df["row_min"]

In [17]:
df = df.replace([np.inf,-np.inf],np.nan)
df = df.fillna(df.median())

In [18]:
train = df[df["is_train"]==1].drop(columns=["is_train"])
test = df[df["is_train"]==0].drop(columns=["is_train","Class"])

X = train.drop("Class",axis=1)
y = train["Class"]

In [19]:
qt = QuantileTransformer(
    n_quantiles=1000,
    output_distribution="normal",
    random_state=42
)

X = pd.DataFrame(qt.fit_transform(X),columns=X.columns)
test = pd.DataFrame(qt.transform(test),columns=test.columns)

In [20]:
pca = PCA(n_components=10,random_state=42)

pca_train = pca.fit_transform(X)
pca_test = pca.transform(test)

for i in range(10):

    X[f"PCA_{i}"] = pca_train[:,i]
    test[f"PCA_{i}"] = pca_test[:,i]

In [21]:
folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(X))
test_preds = np.zeros(len(test))

In [22]:
for fold,(tr,va) in enumerate(folds.split(X,y)):

    X_train,X_val = X.iloc[tr],X.iloc[va]
    y_train,y_val = y.iloc[tr],y.iloc[va]

    lgb_model = lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    xgb_model = XGBClassifier(
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

    cat_model = CatBoostClassifier(
        iterations=1200,
        learning_rate=0.03,
        depth=6,
        verbose=0
    )

    lgb_model.fit(X_train,y_train)
    xgb_model.fit(X_train,y_train)
    cat_model.fit(X_train,y_train)

    lgb_pred = lgb_model.predict_proba(X_val)[:,1]
    xgb_pred = xgb_model.predict_proba(X_val)[:,1]
    cat_pred = cat_model.predict_proba(X_val)[:,1]

    val_pred = 0.4*lgb_pred + 0.4*xgb_pred + 0.2*cat_pred

    oof[va] = val_pred

    print("Fold",fold,"AUC:",roc_auc_score(y_val,val_pred))

    test_lgb = lgb_model.predict_proba(test)[:,1]
    test_xgb = xgb_model.predict_proba(test)[:,1]
    test_cat = cat_model.predict_proba(test)[:,1]

    test_preds += (
        0.4*test_lgb +
        0.4*test_xgb +
        0.2*test_cat
    ) / folds.n_splits

[LightGBM] [Info] Number of positive: 13849, number of negative: 20581
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.063290 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21930
[LightGBM] [Info] Number of data points in the train set: 34430, number of used features: 86
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.402236 -> initscore=-0.396155
[LightGBM] [Info] Start training from score -0.396155
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [23]:
print("Final CV AUC:",roc_auc_score(y,oof))

Final CV AUC: 0.9992038218008562


In [24]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "Class": test_preds
})

In [25]:
submission.to_csv("FINAL.csv",index=False)